In [1]:
import os
import glob
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = "geocorpus"  # ajuste se necessário
REPORTS_DIR = os.path.join(BASE_DIR, "reports_out")
Path(REPORTS_DIR).mkdir(parents=True, exist_ok=True)


# util simples para listar subpastas imediatas
def list_subdirs(path):
    return sorted([p for p in os.listdir(path) if (Path(path) / p).is_dir()])

In [3]:
INS_DIR = os.path.join(BASE_DIR, "insights_out")
splits_insights = list_subdirs(INS_DIR)

In [4]:
def load_insights(base_dir):
    out = {}
    for split in list_subdirs(base_dir):
        sdir = os.path.join(base_dir, split)
        try:
            q = pd.read_csv(os.path.join(sdir, "length_quantiles.csv"))
            q.insert(0, "split", split)
        except FileNotFoundError:
            q = None
        try:
            oov = pd.read_csv(os.path.join(sdir, "oov_rates.csv"))
            oov.insert(0, "split", split)
        except FileNotFoundError:
            oov = None
        try:
            rare = pd.read_csv(os.path.join(sdir, "rare_labels.csv"))
            rare.insert(0, "split", split)
        except FileNotFoundError:
            rare = None
        try:
            tok_te = pd.read_csv(os.path.join(sdir, "tokens_test.csv"))
            tok_te.insert(0, "split", split)
        except FileNotFoundError:
            tok_te = None
        try:
            tok_tr = pd.read_csv(os.path.join(sdir, "tokens_train.csv"))
            tok_tr.insert(0, "split", split)
        except FileNotFoundError:
            tok_tr = None

        out[split] = {
            "quantiles": q,
            "oov": oov,
            "rare": rare,
            "tokens_test": tok_te,
            "tokens_train": tok_tr,
        }
    return out

In [5]:
ins = load_insights(INS_DIR)

In [6]:
# Tabelas consolidadas simples:
ins_quantiles = pd.concat(
    [ins[s]["quantiles"] for s in ins if ins[s]["quantiles"] is not None],
    ignore_index=True,
)
ins_oov = pd.concat(
    [ins[s]["oov"] for s in ins if ins[s]["oov"] is not None], ignore_index=True
)
ins_rare = pd.concat(
    [ins[s]["rare"] for s in ins if ins[s]["rare"] is not None], ignore_index=True
)
ins_tok_te = pd.concat(
    [ins[s]["tokens_test"] for s in ins if ins[s]["tokens_test"] is not None],
    ignore_index=True,
)
ins_tok_tr = pd.concat(
    [ins[s]["tokens_train"] for s in ins if ins[s]["tokens_train"] is not None],
    ignore_index=True,
)

# OOV em formato largo por métrica
ins_oov_wide = ins_oov.pivot(
    index="split", columns="metric", values="value"
).reset_index()

In [7]:
display(ins_oov_wide)

metric,split,oov_rate_test_vs_train,oov_rate_val_vs_train
0,adversarial,0.082947,0.066762
1,heur_len,0.066450,0.061620
2,heur_rare,0.085170,0.058247
3,loc,0.089843,0.083507
4,reverse,0.078687,0.068937
5,semantic,0.091475,0.066158
6,standard,0.059623,0.057993


In [8]:
CD_DIR = os.path.join(BASE_DIR, "class_dist_out")


def load_class_dist(base_dir):
    rows_counts, rows_props, rows_long = [], [], []
    label_vecs_all = []
    for split in list_subdirs(base_dir):
        sdir = os.path.join(base_dir, split)

        # long
        if Path(os.path.join(sdir, "class_distribution_long.csv")).exists():
            df_long = pd.read_csv(os.path.join(sdir, "class_distribution_long.csv"))
            df_long.insert(0, "split_name", split)
            rows_long.append(df_long)

        # counts e props
        if Path(os.path.join(sdir, "counts_pivot.csv")).exists():
            cnt = pd.read_csv(os.path.join(sdir, "counts_pivot.csv"))
            cnt.insert(0, "split_name", split)
            rows_counts.append(cnt)

        if Path(os.path.join(sdir, "props_pivot.csv")).exists():
            pr = pd.read_csv(os.path.join(sdir, "props_pivot.csv"))
            pr.insert(0, "split_name", split)
            rows_props.append(pr)

        # label_vecs.csv (linha por part)
        if Path(os.path.join(sdir, "label_vecs.csv")).exists():
            lv = pd.read_csv(os.path.join(sdir, "label_vecs.csv"))
            lv.insert(0, "split_name", split)
            label_vecs_all.append(lv)

    long_df = pd.concat(rows_long, ignore_index=True) if rows_long else pd.DataFrame()
    counts = (
        pd.concat(rows_counts, ignore_index=True) if rows_counts else pd.DataFrame()
    )
    props = pd.concat(rows_props, ignore_index=True) if rows_props else pd.DataFrame()
    lvecs = (
        pd.concat(label_vecs_all, ignore_index=True)
        if label_vecs_all
        else pd.DataFrame()
    )
    return long_df, counts, props, lvecs


cd_long, cd_counts, cd_props, cd_lv = load_class_dist(CD_DIR)

In [9]:
merged_wide = cd_counts.merge(
    cd_props,
    on=["split_name", "label"],
    suffixes=("_count", "_prop"),
)
merged_wide.to_csv(
    os.path.join(REPORTS_DIR, "class_counts_props_merged_wide.csv"), index=False
)

In [10]:
props_long = cd_props.melt(
    id_vars=["split_name", "label"],
    var_name="part",
    value_name="prop",
)
props_long_sorted = props_long.sort_values(
    ["split_name", "part", "prop"], ascending=[True, True, False]
)

# counts longo para anexar contagem ao top-5
counts_long = cd_counts.melt(
    id_vars=["split_name", "label"],
    var_name="part",
    value_name="count",
)

In [11]:
top5 = (
    props_long_sorted.groupby(["split_name", "part"], group_keys=False)
    .head(5)
    .merge(counts_long, on=["split_name", "label", "part"], how="left")
)

In [12]:
print("== Merged (wide) ==")
display(merged_wide.head())

== Merged (wide) ==


,split_name,label,test_count,train_count,val_count,test_prop,train_prop,val_prop
0,adversarial,B-ambienteSedimentacao,91,45,10,0.002308,0.000406,0.000621
1,adversarial,B-baciaSedimentar,174,332,45,0.004412,0.002993,0.002795
2,adversarial,B-bentonico,6,16,5,0.000152,0.000144,0.000311
3,adversarial,B-campoPetrolifero,0,4,2,0.000000,0.000036,0.000124
4,adversarial,B-constituinteRochaSedimentar,85,24,3,0.002155,0.000216,0.000186


In [13]:
print("\n== Props (long, sorted) ==")
display(props_long_sorted.head(12))


== Props (long, sorted) ==


,split_name,label,part,prop
56,adversarial,O,test,0.908077
26,adversarial,B-sedimentaresSiliciclasticas,test,0.013440
54,adversarial,I-unidadeEstratigrafica,test,0.008064
28,adversarial,B-unidadeEstratigrafica,test,0.006948
5,adversarial,B-contextoGeologicoDeBacia,test,0.005325
1,adversarial,B-baciaSedimentar,test,0.004412
31,adversarial,I-baciaSedimentar,test,0.004286
52,adversarial,I-sedimentaresSiliciclasticas,test,0.004057
23,adversarial,B-sedimentaresCarbonaticas,test,0.003677
10,adversarial,B-estratigrafia,test,0.003525


In [14]:
print("\n== Top-5 por split e partição ==")
display(top5)


== Top-5 por split e partição ==


,split_name,label,part,prop,count
0,adversarial,O,test,0.908077,35810
1,adversarial,B-sedimentaresSiliciclasticas,test,0.013440,530
2,adversarial,I-unidadeEstratigrafica,test,0.008064,318
3,adversarial,B-unidadeEstratigrafica,test,0.006948,274
4,adversarial,B-contextoGeologicoDeBacia,test,0.005325,210
...,...,...,...,...,...
100,standard,O,val,0.929312,15592
101,standard,B-idade,val,0.005305,89
102,standard,B-sedimentaresSiliciclasticas,val,0.005245,88
103,standard,B-epoca,val,0.005066,85


In [15]:
top5['part'].value_counts()

part
test     35
train    35
val      35
Name: count, dtype: int64

In [16]:
parts = ["train", "val", "test"]

for split in sorted(top5["split_name"].unique()):
    print(f"\n=== {split} ===")
    for part in parts:
        sub = (
            top5[(top5["split_name"] == split) & (top5["part"] == part)]
            .sort_values("prop", ascending=False)
            .head(5)
            .loc[:, ["label", "prop", "count"]]
            .reset_index(drop=True)
        )
        print(f"\n[{part}] top-5")
        try:
            display(sub)  # funciona no Jupyter
        except NameError:
            print(sub.to_string(index=False))  # fallback se display não existir


=== adversarial ===

[train] top-5


,label,prop,count
0,O,0.934442,103653
1,B-idade,0.005598,621
2,B-periodo,0.004913,545
3,B-epoca,0.004733,525
4,I-unidadeEstratigrafica,0.004715,523



[val] top-5


,label,prop,count
0,O,0.932679,15018
1,B-sedimentaresSiliciclasticas,0.005900,95
2,B-magmaticas,0.005651,91
3,B-idade,0.004844,78
4,B-periodo,0.004844,78



[test] top-5


...

,label,prop,count
0,O,0.927914,123253
1,B-sedimentaresSiliciclasticas,0.006505,864
2,I-unidadeEstratigrafica,0.005631,748
3,B-idade,0.004630,615
4,B-unidadeEstratigrafica,0.004630,615



[val] top-5


,label,prop,count
0,O,0.929312,15592
1,B-idade,0.005305,89
2,B-sedimentaresSiliciclasticas,0.005245,88
3,B-epoca,0.005066,85
4,B-contextoGeologicoDeBacia,0.004530,76



[test] top-5


,label,prop,count
0,O,0.927622,15636
1,B-sedimentaresSiliciclasticas,0.008662,146
2,B-idade,0.005458,92
3,I-unidadeEstratigrafica,0.004924,83
4,B-unidadeEstratigrafica,0.004746,80


In [17]:
CWI_DIR = os.path.join(BASE_DIR, "cosine_out")

# tenta usar o resumo pronto; se não existir, empilha de cada split
summary_path = os.path.join(CWI_DIR, "cosine_summary_all_splits.csv")
if Path(summary_path).exists():
    cos_within_all = pd.read_csv(summary_path)
else:
    rows = []
    for split in list_subdirs(CWI_DIR):
        f = os.path.join(CWI_DIR, split, "cosine_all.csv")
        if Path(f).exists():
            df = pd.read_csv(f)
            df.insert(0, "split", split)
            rows.append(df)
    cos_within_all = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

# Tabelas simples:
# - matriz (a,b) por espaço+split_set (labels/words e pares)
if not cos_within_all.empty:
    cos_within_pivot = (
        cos_within_all.assign(pair=lambda d: d["a"] + "_" + d["b"])
        .pivot_table(
            index=["split", "space"],
            columns="pair",
            values="cosine_distance",
            aggfunc="first",
        )
        .reset_index()
    )
else:
    cos_within_pivot = pd.DataFrame()

In [18]:
cos_within_all.query("space == 'words'")[['split', 'a', 'b', 'cosine_distance']]

,split,a,b,cosine_distance
3,standard,train,val,0.067678
4,standard,train,test,0.068365
5,standard,val,test,0.114670
9,heur_len,train,val,0.063732
10,heur_len,train,test,0.040181
11,heur_len,val,test,0.081791
15,heur_rare,train,val,0.068075
16,heur_rare,train,test,0.046645
17,heur_rare,val,test,0.083270
21,adversarial,train,val,0.069274


In [19]:
cos_within_all.query("space == 'labels'")[["split", "a", "b", "cosine_distance"]]

,split,a,b,cosine_distance
0,standard,train,val,0.000005
1,standard,train,test,0.000007
2,standard,val,test,0.000014
6,heur_len,train,val,0.000005
7,heur_len,train,test,0.000002
8,heur_len,val,test,0.000004
12,heur_rare,train,val,0.000005
13,heur_rare,train,test,0.000004
14,heur_rare,val,test,0.000008
18,adversarial,train,val,0.000008


In [20]:
display(cos_within_pivot)

pair,split,space,train_test,train_val,val_test
0,adversarial,labels,0.000118,0.000008,0.000110
1,adversarial,words,0.129919,0.069274,0.157724
2,heur_len,labels,0.000002,0.000005,0.000004
3,heur_len,words,0.040181,0.063732,0.081791
4,heur_rare,labels,0.000004,0.000005,0.000008
5,heur_rare,words,0.046645,0.068075,0.083270
6,loc,labels,0.000039,0.000044,0.000050
7,loc,words,0.105699,0.161427,0.177469
8,reverse,labels,0.000329,0.000047,0.000166
9,reverse,words,0.133647,0.101504,0.135467


In [21]:
cos_within_pivot.query("space == 'labels'")[
    ["split", "train_test", "train_val", "val_test"]
]

pair,split,train_test,train_val,val_test
0,adversarial,0.000118,0.000008,0.000110
2,heur_len,0.000002,0.000005,0.000004
4,heur_rare,0.000004,0.000005,0.000008
6,loc,0.000039,0.000044,0.000050
8,reverse,0.000329,0.000047,0.000166
10,semantic,0.000109,0.000075,0.000219
12,standard,0.000007,0.000005,0.000014


In [22]:
cos_within_pivot.query("space == 'words'")[
    ["split", "train_test", "train_val", "val_test"]
]

pair,split,train_test,train_val,val_test
1,adversarial,0.129919,0.069274,0.157724
3,heur_len,0.040181,0.063732,0.081791
5,heur_rare,0.046645,0.068075,0.083270
7,loc,0.105699,0.161427,0.177469
9,reverse,0.133647,0.101504,0.135467
11,semantic,0.313795,0.165815,0.413728
13,standard,0.068365,0.067678,0.114670


In [23]:
CB_DIR = os.path.join(BASE_DIR, "cosine_between_out")


def safe_read_csv(path):
    return pd.read_csv(path) if Path(path).exists() else None


cos_bw_train_words = safe_read_csv(os.path.join(CB_DIR, "cos_train_words.csv"))
cos_bw_test_words = safe_read_csv(os.path.join(CB_DIR, "cos_test_words.csv"))
cos_bw_train_labels = safe_read_csv(os.path.join(CB_DIR, "cos_train_labels.csv"))
cos_bw_test_labels = safe_read_csv(os.path.join(CB_DIR, "cos_test_labels.csv"))

In [24]:
cos_bw_train_words

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000,0.002493,0.010250,0.015372,0.012072,0.031638,0.020117
1,heur_len,0.002493,0.000000,0.012634,0.018225,0.014582,0.033932,0.022602
2,heur_rare,0.010250,0.012634,0.000000,0.016709,0.013122,0.039957,0.023860
3,adversarial,0.015372,0.018225,0.016709,0.000000,0.021583,0.051913,0.022633
4,loc,0.012072,0.014582,0.013122,0.021583,0.000000,0.039488,0.032270
5,semantic,0.031638,0.033932,0.039957,0.051913,0.039488,0.000000,0.050818
6,reverse,0.020117,0.022602,0.023860,0.022633,0.032270,0.050818,0.000000


In [25]:
cos_bw_test_words

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000,0.102337,0.161103,0.185166,0.218516,0.320098,0.184596
1,heur_len,0.102337,0.000000,0.108726,0.152113,0.168419,0.265876,0.144382
2,heur_rare,0.161103,0.108726,0.000000,0.149234,0.147963,0.322530,0.137365
3,adversarial,0.185166,0.152113,0.149234,0.000000,0.239864,0.419941,0.122109
4,loc,0.218516,0.168419,0.147963,0.239864,0.000000,0.337993,0.229912
5,semantic,0.320098,0.265876,0.322530,0.419941,0.337993,0.000000,0.407597
6,reverse,0.184596,0.144382,0.137365,0.122109,0.229912,0.407597,0.000000


In [26]:
cos_bw_train_labels

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000,0.000004,0.000006,0.993632,0.000012,0.000009,0.999862
1,heur_len,0.000004,0.000000,0.000011,0.993525,0.000016,0.000004,0.999855
2,heur_rare,0.000006,0.000011,0.000000,0.993660,0.000019,0.000014,0.999881
3,adversarial,0.993632,0.993525,0.993660,0.000000,0.993368,0.992506,0.996923
4,loc,0.000012,0.000016,0.000019,0.993368,0.000000,0.000020,0.999867
5,semantic,0.000009,0.000004,0.000014,0.992506,0.000020,0.000000,0.999839
6,reverse,0.999862,0.999855,0.999881,0.996923,0.999867,0.999839,0.000000


In [27]:
cos_bw_test_labels

,Unnamed: 0,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
0,standard,0.000000,0.000013,0.000015,0.995064,0.000044,0.000100,0.999333
1,heur_len,0.000013,0.000000,0.000010,0.994867,0.000033,0.000075,0.999396
2,heur_rare,0.000015,0.000010,0.000000,0.994797,0.000028,0.000086,0.999316
3,adversarial,0.995064,0.994867,0.994797,0.000000,0.995369,0.998543,0.994778
4,loc,0.000044,0.000033,0.000028,0.995369,0.000000,0.000050,0.999393
5,semantic,0.000100,0.000075,0.000086,0.998543,0.000050,0.000000,0.999528
6,reverse,0.999333,0.999396,0.999316,0.994778,0.999393,0.999528,0.000000
